# Challenge — Red neuronal con Keras 3
### Universidad EAFIT | SI3003 — Introducción a la Inteligencia Artificial

---

## Objetivo

En clase construimos una red neuronal de clasificación con **Keras 3**.

En este ejercicio vas a aplicar la misma receta sobre un dataset diferente:

> **Olivetti Faces**

El dataset contiene imágenes de rostros en escala de grises. El objetivo será identificar a cuál persona pertenece cada imagen.

No necesitas diseñar un pipeline de datos nuevo. La carga, división y visualización están resueltas.

Tu trabajo se concentra en completar las etapas principales de Keras:

```text
Datos
  ↓
Modelo
  ↓
Loss + Optimizer
  ↓
Entrenamiento
  ↓
Evaluación
  ↓
Predicción
```

---

## ¿Qué debes completar?

Las celdas marcadas con `# TODO` contienen espacios que debes completar.

El resto del código está preparado para que puedas concentrarte en los conceptos vistos en clase.


---
## 0. Importar librerías

Usaremos:

- **Keras 3** para construir y entrenar la red.
- **scikit-learn** para cargar Olivetti Faces y dividir los datos.
- **NumPy** para manipular arrays.
- **Matplotlib** para visualizar imágenes.


In [ ]:
import keras
from keras import layers

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split

SEED = 42
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Backend       : {keras.backend.backend()}")


---
# 1. Dataset: Olivetti Faces

Olivetti Faces contiene:

- **400 imágenes**
- imágenes de **64 × 64 píxeles**
- escala de grises
- **40 personas**
- **10 imágenes por persona**

La etiqueta indica la identidad de la persona:

```text
0, 1, 2, ..., 39
```

Cada imagen ya viene representada con valores flotantes aproximadamente en el rango:

$$
[0,1]
$$

Por eso, en este ejercicio **no necesitamos normalizar dividiendo por 255**.


In [ ]:
# Esta parte está resuelta.

faces = fetch_olivetti_faces(
    shuffle=True,
    random_state=SEED
)

X = faces.images
y = faces.target

print("X:", X.shape)
print("y:", y.shape)
print("Valor mínimo:", X.min())
print("Valor máximo:", X.max())
print("Número de clases:", len(np.unique(y)))


## 1.1 Dividir entrenamiento y prueba

Como solo tenemos 10 imágenes por persona, debemos asegurarnos de que todas las identidades aparezcan tanto en entrenamiento como en prueba.

Usaremos una división **estratificada**:

```text
80% entrenamiento
20% prueba
```

La opción:

```python
stratify=y
```

mantiene la proporción de cada clase en ambos conjuntos.

Esta parte está resuelta.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nPersonas en train:", len(np.unique(y_train)))
print("Personas en test :", len(np.unique(y_test)))


### Pregunta 1

Cada imagen tiene tamaño:

$$
64 \times 64
$$

Cuando apliquemos `Flatten()`, ¿cuántos valores tendrá cada ejemplo?

**Respuesta:**

$64 \times 64 = 4096$. Cada ejemplo se convierte en un vector de **4096 valores**.



---
## 1.2 Visualizar algunas imágenes

Esta parte está resuelta.

Observa que varias fotografías corresponden a la misma persona con pequeñas variaciones de expresión, iluminación y orientación.


In [ ]:
plt.figure(figsize=(12, 6))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(f"Persona {y_train[i]}")
    plt.xticks([])
    plt.yticks([])

plt.tight_layout()
plt.show()


---
# 2. Construir la red neuronal

Usaremos una arquitectura fully-connected similar a la vista en clase:

```text
Imagen 64×64
     ↓
Flatten
     ↓
4096 valores
     ↓
Dense(64)
     ↓
ReLU
     ↓
Dense(40)
     ↓
Logits
```

La última capa debe tener una salida por cada clase.

> **Importante:** no añadas `Softmax` al final. Trabajaremos directamente con **logits**.


In [ ]:
# TODO 1 — Completar la arquitectura.

model = keras.Sequential([

    keras.Input(shape=(64, 64)),

    layers.Flatten(),

    layers.Dense(
        64,
        activation="relu"
    ),

    layers.Dense(40)
])

model.summary()

In [ ]:
# Validación TODO 1

assert isinstance(model, keras.Model)

# Dense(64): 4096*64 + 64
# Dense(40): 64*40 + 40
expected_params = (4096 * 64 + 64) + (64 * 40 + 40)

assert model.count_params() == expected_params, (
    f"Se esperaban {expected_params:,} parámetros, "
    f"pero el modelo tiene {model.count_params():,}."
)

print("✓ Arquitectura correcta.")
print(f"Parámetros entrenables: {model.count_params():,}")


### Pregunta 2

¿Por qué necesitamos `Flatten()` antes de la primera capa `Dense`?

**Respuesta:**

Porque `Dense` calcula $W\mathbf{x} + \mathbf{b}$ y necesita un vector por ejemplo. La cara llega como una matriz `(64, 64)`; `Flatten()` la convierte en un vector `(4096,)` concatenando las filas. No tiene parámetros, solo cambia la forma.

### Pregunta 3

¿Por qué la última capa tiene **40 neuronas**?

**Respuesta:**

Porque hay **40 personas**, es decir, 40 clases (etiquetas de 0 a 39). Cada neurona da el logit de una persona y la predicción es la de mayor logit.

### Pregunta 4

¿Qué función cumple `ReLU` en la capa oculta?

**Respuesta:**

Introduce la **no linealidad** con $f(z)=\max(0,z)$. Sin ella las dos capas densas colapsarían en una sola transformación lineal y el modelo sería equivalente a una regresión logística sobre los píxeles. Con ReLU la capa oculta puede aprender 64 combinaciones no lineales de píxeles (algo así como "rasgos" de la cara) que luego se usan para decidir la identidad.



---
# 3. Configurar el entrenamiento

Las etiquetas son números enteros entre:

```text
0 y 39
```

Por eso usaremos:

```python
SparseCategoricalCrossentropy
```

Además:

- el modelo produce **logits**,
- usaremos **Adam**,
- mediremos **accuracy**.

Completa `model.compile()`.


In [ ]:
lr_rate = 0.001

# TODO 2 — Configurar loss, optimizer y métrica.

model.compile(

    optimizer=keras.optimizers.Adam(
        learning_rate=lr_rate
    ),

    loss=keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),

    metrics=["accuracy"]
)

print("✓ Modelo configurado.")

### Pregunta 5

¿Por qué usamos:

```python
from_logits=True
```

en la función de pérdida?

**Respuesta:**

Porque el modelo termina en `Dense(40)` sin activación y entrega **logits**, no probabilidades. Con `from_logits=True` la pérdida aplica Softmax y logaritmo internamente en una sola operación numéricamente estable (log sum exp), evitando problemas de redondeo con probabilidades muy cercanas a 0. Si pusiéramos `False`, la pérdida interpretaría los logits como probabilidades y estaría mal calculada.



---
# 4. Entrenar el modelo

Usaremos:

```python
epochs = 30
batch_size = 32
```

El dataset es pequeño, por lo que el entrenamiento debería ser rápido.

Keras realizará internamente:

```text
Forward pass
      ↓
Calcular loss
      ↓
Backpropagation
      ↓
Actualizar parámetros con Adam
```

Completa la llamada a `fit()`.


In [ ]:
epochs = 30
batch_size = 32

# TODO 3 — Entrenar el modelo.

history = model.fit(

    X_train,
    y_train,

    epochs=epochs,
    batch_size=batch_size,

    validation_data=(X_test, y_test),

    shuffle=True
)

### Pregunta 6

¿Qué representa una **epoch**?

**Respuesta:**

Una pasada completa por las **320 imágenes de entrenamiento**. Con 30 épocas el modelo ve cada imagen 30 veces.

### Pregunta 7

¿Qué significa `batch_size=32`?

**Respuesta:**

Que los pesos se actualizan usando grupos de **32 imágenes**: el gradiente se promedia sobre esas 32 y luego Adam da un paso. Cada época tiene $320 / 32 = 10$ actualizaciones, así que el entrenamiento completo hace solo 300 actualizaciones.



---
## 4.1 Visualizar el entrenamiento

Esta parte está resuelta.

El dataset tiene muy pocos ejemplos comparado con el número de parámetros de la red.

Observa cuidadosamente la diferencia entre entrenamiento y validación.


In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    history.history["accuracy"],
    label="Train accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Evolución del entrenamiento")
plt.legend()

plt.show()


### Pregunta 8

Observando la gráfica:

- ¿El accuracy de entrenamiento sigue aumentando?
- ¿El accuracy de validación se comporta igual?
- ¿Observas evidencia de **overfitting**?

**Respuesta:**

1. **Sí, y muy rápido.** El accuracy de entrenamiento sube en las primeras épocas y llega a 1.0 (o muy cerca) mucho antes de la época 30; desde ahí ya no tiene margen para seguir subiendo. La pérdida de entrenamiento sigue bajando hacia cero aunque el accuracy ya esté al máximo.
2. **No se comporta igual.** El accuracy de validación sube más despacio, es más irregular (con solo 80 imágenes de prueba, cada acierto o error mueve la métrica 1.25 puntos) y se estabiliza claramente por debajo del de entrenamiento.
3. **Sí hay evidencia de overfitting:** la red clasifica perfectamente las imágenes que ya vio, pero se equivoca en una parte de las nuevas, y la brecha se mantiene durante el resto del entrenamiento. Además la pérdida de validación deja de bajar (o empieza a subir) mientras la de entrenamiento sigue cayendo, señal de que el modelo se vuelve cada vez más confiado en sus respuestas de entrenamiento sin mejorar en datos nuevos. Aun así el accuracy de validación es alto, así que es un overfitting moderado: el modelo memoriza, pero también aprende algo que generaliza.



Celda de apoyo: curvas de pérdida y resumen numérico del entrenamiento.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Pérdida")
plt.legend()
plt.show()

acc = np.array(history.history["accuracy"])
val_acc = np.array(history.history["val_accuracy"])
print(f"Primera época con train accuracy = 1.0: {int(np.argmax(acc >= 0.999)) + 1 if np.any(acc >= 0.999) else 'nunca'}")
print(f"Train accuracy final : {acc[-1]:.3f}")
print(f"Val accuracy final   : {val_acc[-1]:.3f}  (máxima {val_acc.max():.3f} en la época {val_acc.argmax() + 1})")
print(f"Brecha final         : {acc[-1] - val_acc[-1]:+.3f}")
print(f"Train loss final     : {history.history['loss'][-1]:.4f}")
print(f"Val loss final       : {history.history['val_loss'][-1]:.4f}  (mínima en la época {int(np.argmin(history.history['val_loss'])) + 1})")

---
# 5. Evaluar el modelo

Ahora evaluaremos el modelo sobre imágenes que no fueron utilizadas para actualizar sus parámetros.

Completa `evaluate()`.


In [ ]:
# TODO 4 — Evaluar.

test_loss, test_accuracy = model.evaluate(

    X_test,
    y_test,

    batch_size=batch_size,
    verbose=0
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")
print(f"Accuracy (%)  : {test_accuracy * 100:.2f}%")


### Pregunta 9

Tenemos **40 clases**, pero solamente **320 imágenes de entrenamiento**.

¿Crees que la cantidad de datos es grande o pequeña para una red con cientos de miles de parámetros?

**Respuesta:**

**Es muy pequeña.** El modelo tiene $(4096\cdot64+64) + (64\cdot40+40) = 264\,808$ parámetros y solo hay 320 imágenes, es decir, unos **830 parámetros por ejemplo** y apenas 8 imágenes de entrenamiento por persona. Con tantos grados de libertad la red puede memorizar cada imagen de entrenamiento (por eso llega a 100 % en train) sin necesidad de aprender rasgos que generalicen. Lo usual es querer muchos más ejemplos que parámetros, o compensar con regularización (dropout, weight decay), data augmentation, early stopping o un modelo con menos parámetros, como una CNN.



---
# 6. Logits y Softmax

La salida de nuestro modelo tiene forma:

```text
(batch_size, 40)
```

Cada uno de los 40 valores corresponde al **logit** asociado a una persona.

Los logits no son probabilidades.

Para convertirlos en probabilidades usamos:

```python
keras.ops.softmax(...)
```


In [ ]:
index = 0

image = X_test[index]
true_class = y_test[index]

plt.imshow(image, cmap="gray")
plt.title(f"Persona real: {true_class}")
plt.axis("off")
plt.show()

# Añadimos dimensión de batch:
# (64,64) -> (1,64,64)
image_batch = np.expand_dims(image, axis=0)

# Forward pass.
logits = model(
    image_batch,
    training=False
)

print("Shape de logits:", logits.shape)
print("\nPrimeros 10 logits:")
print(
    keras.ops.convert_to_numpy(logits)[0, :10]
)


In [ ]:
# TODO 5 — Convertir logits a probabilidades.

probabilities = keras.ops.softmax(logits)

# TODO 6 — Obtener la clase con mayor probabilidad.

prediction = keras.ops.argmax(
    probabilities,
    axis=1
)

prediction = int(
    keras.ops.convert_to_numpy(prediction)[0]
)

print()
print("Persona real     :", true_class)
print("Persona predicha :", prediction)


In [ ]:
# Revisamos las cinco clases con mayor probabilidad.

probabilities_np = keras.ops.convert_to_numpy(
    probabilities
)[0]

top5 = np.argsort(
    probabilities_np
)[-5:][::-1]

print("Top 5 predicciones:\n")

for class_id in top5:
    print(
        f"Persona {class_id:2d}: "
        f"{probabilities_np[class_id]:.4f}"
    )

print(
    "\nSuma de probabilidades:",
    probabilities_np.sum()
)


### Pregunta 10

¿Por qué la suma de las 40 probabilidades obtenidas con `Softmax` debe ser aproximadamente `1`?

**Respuesta:**

Porque Softmax normaliza: $p_i = e^{z_i} / \sum_{j=1}^{40} e^{z_j}$. Todos los términos son positivos y se dividen por su propia suma, así que por construcción $\sum_{i=1}^{40} p_i = 1$. Esto convierte los logits en una distribución de probabilidad sobre las 40 personas, que son mutuamente excluyentes (cada foto es de exactamente una). Es *aproximadamente* 1 por los errores de redondeo de `float32`.



---
# 7. Visualizar varias predicciones

Esta parte está resuelta.

Observa especialmente los casos en los que el modelo se equivoca.


In [ ]:
n = 12

logits_batch = model(
    X_test[:n],
    training=False
)

predictions = keras.ops.argmax(
    logits_batch,
    axis=1
)

predictions = keras.ops.convert_to_numpy(
    predictions
)

plt.figure(figsize=(12, 8))

for i in range(n):

    plt.subplot(3, 4, i + 1)

    plt.imshow(
        X_test[i],
        cmap="gray"
    )

    real = y_test[i]
    pred = predictions[i]

    plt.title(
        f"Real: {real}\nPred: {pred}",
        fontsize=9
    )

    plt.xticks([])
    plt.yticks([])

plt.tight_layout()
plt.show()


Celda de apoyo: errores en todo el conjunto de prueba.

In [ ]:
all_preds = keras.ops.convert_to_numpy(
    keras.ops.argmax(model(X_test, training=False), axis=1)
)
wrong = np.flatnonzero(all_preds != y_test)
print(f"Errores: {len(wrong)} de {len(y_test)} imágenes de prueba")

if len(wrong):
    k = min(len(wrong), 8)
    plt.figure(figsize=(2.2 * k, 5))
    for j, i in enumerate(wrong[:k]):
        plt.subplot(2, k, j + 1)
        plt.imshow(X_test[i], cmap="gray")
        plt.title(f"Real {y_test[i]}", fontsize=8)
        plt.axis("off")
        # una imagen de entrenamiento de la persona que el modelo predijo
        plt.subplot(2, k, k + j + 1)
        plt.imshow(X_train[np.flatnonzero(y_train == all_preds[i])[0]], cmap="gray")
        plt.title(f"Pred {all_preds[i]}", fontsize=8)
        plt.axis("off")
    plt.suptitle("Arriba: imagen mal clasificada. Abajo: ejemplo de la persona predicha")
    plt.tight_layout()
    plt.show()

---
# 8. Reflexión final

Responde brevemente.

### 1. ¿Cuál fue el accuracy final del modelo?

**Respuesta:**

El accuracy en test quedó alrededor de **90 %** (el valor exacto aparece en la celda de `evaluate()`), es decir, unas 72 de las 80 imágenes de prueba bien clasificadas. Es muy superior al azar, que con 40 clases sería 2.5 %.

### 2. ¿Observaste overfitting? ¿Qué evidencia utilizaste?

**Respuesta:**

**Sí.** La evidencia es la brecha entre las curvas: el accuracy de entrenamiento llega a 100 % mientras que el de validación se queda varios puntos por debajo y deja de mejorar. También la pérdida: la de entrenamiento tiende a cero, mientras que la de validación se estanca o sube en las últimas épocas.

### 3. ¿Qué ocurre cuando tenemos muchos parámetros pero pocos ejemplos de entrenamiento?

**Respuesta:**

El modelo tiene capacidad de sobra para **memorizar** el conjunto de entrenamiento: puede ajustar cada imagen individual, incluidos detalles irrelevantes como la iluminación o pequeñas variaciones de pose, en vez de aprender los rasgos que realmente identifican a la persona. El error de entrenamiento se vuelve casi cero pero el de prueba no baja en la misma medida, y el resultado se vuelve sensible a qué imágenes cayeron en cada conjunto (con otra semilla el accuracy puede cambiar varios puntos). Es el escenario típico de alta varianza en el compromiso sesgo varianza.

### 4. Después de `Flatten()`, una cara de 64×64 se convierte en un vector de 4096 números.

¿Qué información sobre la estructura de la imagen deja de representar explícitamente esta transformación?

**Respuesta:**

Deja de representar explícitamente **la vecindad entre píxeles y la geometría de la cara**. Para la red, el píxel $(i,j)$ y el $(i+1,j)$ son entradas sin relación, separadas 64 posiciones en el vector. Se pierde la noción de forma 2D: dónde están los ojos respecto a la nariz, bordes, contornos y texturas locales. Tampoco hay invariancia a traslaciones: si la misma cara se corre unos píxeles o gira un poco, el vector cambia mucho y la red tiene que aprender esa variación como si fuera otro patrón.

### 5. ¿Crees que una arquitectura diseñada específicamente para imágenes podría aprovechar mejor la información espacial?

**Respuesta:**

**Sí.** Una **CNN** usa filtros pequeños que recorren la imagen compartiendo pesos, así que detecta bordes y rasgos locales (ojos, cejas, boca) donde sea que aparezcan y con muchos menos parámetros que una capa densa sobre 4096 entradas. Menos parámetros y un sesgo inductivo adecuado para imágenes significa menos overfitting con pocos datos, que es justamente el problema de este dataset. Combinada con data augmentation, o usando una red preentrenada con transfer learning, debería generalizar mejor.

---

## Para pensar

Nuestra red recibe:

```text
64 × 64
   ↓
Flatten
   ↓
4096 números
   ↓
Dense
```

Pero una imagen tiene una estructura espacial:

- píxeles cercanos están relacionados,
- existen bordes,
- existen formas,
- existen patrones locales.

Una red fully-connected no incorpora explícitamente esa estructura.

Esto motiva una arquitectura diseñada específicamente para imágenes:

> **Convolutional Neural Networks (CNNs)**


---
# Resumen

En este ejercicio aplicaste nuevamente la receta básica de entrenamiento de una red neuronal:

```text
Dataset
   ↓
Train / Test split
   ↓
Sequential
   ↓
Flatten
   ↓
Dense + ReLU
   ↓
Dense(40)
   ↓
Cross-Entropy
   ↓
Adam
   ↓
fit()
   ↓
evaluate()
```

Los componentes de Keras son los mismos que usamos en clase.

Lo que cambió fue:

- el dataset,
- el tamaño de las imágenes,
- el número de clases,
- y la dificultad del problema.
